# Bio-Rad Gate Model: Training and Prediction

Run the cells in order with the `learn_python` kernel. This notebook reproduces the training and prediction scripts with a grouped holdout and 400-tree Random Forest.

Reference labels are ClusterData-mean midpoints or amplitude-bound estimates, not original saved manual gate coordinates. Estimated labels are included in training and evaluation. All-well predictions include training wells and must not be used as independent test results. Saved model and CSV artifacts are overwritten.

## 1. Import Dependencies and Define Paths

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline

PROJECT_DIR = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / 'Biorad_manual_gating').is_dir()
     and (candidate / 'data_output').is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Open this notebook from within the project directory.')
OUTPUT_DIR = PROJECT_DIR / 'model_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUTPUT_DIR / 'biorad_gate_model.joblib'
METRICS_PATH = OUTPUT_DIR / 'biorad_gate_model_metrics.json'
TEST_PREDICTIONS_PATH = OUTPUT_DIR / 'biorad_gate_model_predictions.csv'
IMPORTANCE_PATH = OUTPUT_DIR / 'biorad_gate_feature_importance.csv'
PREDICTION_OUTPUT_PATH = OUTPUT_DIR / 'biorad_gate_predictions_all_wells.csv'
TEST_SIZE = 0.2
RANDOM_STATE = 42
print(f'Project: {PROJECT_DIR}\nOutput: {OUTPUT_DIR}')

Project: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating
Output: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\output


In [ ]:
"""2. Constants and reusable helpers."""
TARGET_COLUMNS = ['x_gate', 'y_gate']
GROUP_COLUMN = 'parent_dataset'
IDENTIFIER_COLUMNS = ['dataset', 'parent_dataset', 'well']
LEAKED_COLUMNS = {
    'x_gate', 'y_gate', 'target1(MUT)', 'target2(WT)', 'quadrant',
    'count', 'ch1_mean', 'ch2_mean', 'gate_has_x', 'gate_has_y',
    'gate_n_cluster_rows', 'gate_total_cluster_count',
    'gate_ch1_mean_weighted', 'gate_ch2_mean_weighted',
    'n_populated_quadrants', 'weighted_mean_ch1_MUT', 'weighted_mean_ch2_WT',
}


def default_data_path() -> Path:
    return PROJECT_DIR / 'data_output' / 'full_biorad_cluster_dataset.csv'


def select_feature_columns(data: pd.DataFrame) -> list[str]:
    excluded = LEAKED_COLUMNS | set(IDENTIFIER_COLUMNS)
    return [column for column in data.columns
            if column not in excluded and pd.api.types.is_numeric_dtype(data[column])]


DATA_PATH = default_data_path()
INFERENCE_DATA_PATH = DATA_PATH
print(f'Training input: {DATA_PATH}\nInference input: {INFERENCE_DATA_PATH}')

Training input: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\output\full_biorad_cluster_dataset.csv
Inference input: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\output\full_biorad_cluster_dataset.csv


In [3]:
"""3. Load and validate training data."""
data = pd.read_csv(DATA_PATH)
required_columns = TARGET_COLUMNS + IDENTIFIER_COLUMNS
missing_columns = [column for column in required_columns if column not in data]
if missing_columns:
    raise ValueError(f'Required columns are missing: {missing_columns}')
input_rows = len(data)
data[TARGET_COLUMNS] = data[TARGET_COLUMNS].apply(pd.to_numeric, errors='coerce')
data = data.dropna(subset=TARGET_COLUMNS + [GROUP_COLUMN]).copy()
print(f'Input wells: {input_rows}; eligible wells: {len(data)}; dropped: {input_rows - len(data)}')
method_columns = [column for column in ['x_gate_method', 'y_gate_method'] if column in data]
for column in method_columns:
    display(data[column].value_counts(dropna=False).rename_axis(column).to_frame('wells'))
display(data.head())

Input wells: 8713; eligible wells: 8713; dropped: 0


,wells
x_gate_method,
cluster_means,8532
single_observed_class_amplitude_bound,181


,wells
y_gate_method,
cluster_means,6377
single_observed_class_amplitude_bound,2336


,dataset,parent_dataset,well,x_gate,y_gate,x_gate_method,y_gate_method,x_min_ch2,x_max_ch2,y_min_ch1,...,ch2_iqr,ch2_cv,ch2_skew,ch2_kurt,ch2_bimodality,ch2_log_std,ch2_log_skew,ch2_outlier_frac,ch2_kde_peaks,ch1_ch2_corr
0,Rapeseed TRT-13_TRT001_A01_Amplitude,Rapeseed TRT-13_TRT001,A01,1525.437550,3438.880518,cluster_means,cluster_means,362.2588,3112.239,300.7066,...,1281.36400,0.384056,-0.022360,-1.844808,0.866090,0.410681,-0.098693,0.0,2,0.395736
1,Rapeseed TRT-13_TRT001_A02_Amplitude,Rapeseed TRT-13_TRT001,A02,1658.720934,2724.028515,cluster_means,cluster_means,834.7704,3166.599,1662.3340,...,1283.39850,0.338357,-0.420272,-1.586671,0.832523,0.387740,-0.523402,0.0,2,0.373964
2,Rapeseed TRT-13_TRT001_A03_Amplitude,Rapeseed TRT-13_TRT001,A03,1686.486856,4119.919418,cluster_means,cluster_means,844.3401,3003.538,1668.9320,...,1246.63150,0.329829,-0.493250,-1.522239,0.841337,0.381758,-0.594342,0.0,2,0.337172
3,Rapeseed TRT-13_TRT001_A04_Amplitude,Rapeseed TRT-13_TRT001,A04,1669.179179,2809.077206,cluster_means,cluster_means,847.4105,3176.465,1648.6370,...,335.33350,0.246442,-1.133239,-0.008224,0.763503,0.305406,-1.375745,0.0,2,0.256095
4,Rapeseed TRT-13_TRT001_A05_Amplitude,Rapeseed TRT-13_TRT001,A05,1755.753028,2688.398641,cluster_means,cluster_means,864.9615,3054.688,1698.9040,...,1248.86925,0.359014,-0.116416,-1.788716,0.836759,0.388903,-0.206399,0.0,2,0.317222


In [4]:
"""4. Select leakage-resistant numeric features."""
feature_columns = select_feature_columns(data)
if not feature_columns:
    raise ValueError('No numeric, non-leaking feature columns were found.')
assert not LEAKED_COLUMNS.intersection(feature_columns)
print(f'Selected {len(feature_columns)} features:')
print(feature_columns)

Selected 28 features:
['x_min_ch2', 'x_max_ch2', 'y_min_ch1', 'y_max_ch1', 'amplitude_range_WT', 'amplitude_range_MUT', 'n_droplets', 'ch1_std', 'ch1_iqr', 'ch1_cv', 'ch1_skew', 'ch1_kurt', 'ch1_bimodality', 'ch1_log_std', 'ch1_log_skew', 'ch1_outlier_frac', 'ch1_kde_peaks', 'ch2_std', 'ch2_iqr', 'ch2_cv', 'ch2_skew', 'ch2_kurt', 'ch2_bimodality', 'ch2_log_std', 'ch2_log_skew', 'ch2_outlier_frac', 'ch2_kde_peaks', 'ch1_ch2_corr']


In [5]:
"""5. Group-aware train/test split."""
groups = data[GROUP_COLUMN].astype(str)
if groups.nunique() < 2:
    raise ValueError('At least two parent_dataset groups are required for evaluation.')
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(data[feature_columns], data[TARGET_COLUMNS], groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
display(pd.DataFrame({
    'split': ['train', 'test'],
    'rows': [len(train_idx), len(test_idx)],
    'groups': [groups.iloc[train_idx].nunique(), groups.iloc[test_idx].nunique()],
}))

,split,rows,groups
0,train,6895,73
1,test,1818,19


In [6]:
"""6. Train the multi-output Random Forest pipeline."""
model = make_pipeline(
    SimpleImputer(strategy='median'),
    RandomForestRegressor(
        n_estimators=400, min_samples_leaf=3, max_features=0.8,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
)
model.fit(data.iloc[train_idx][feature_columns], data.iloc[train_idx][TARGET_COLUMNS])
test_predictions = model.predict(data.iloc[test_idx][feature_columns])
print(f'Trained on {len(train_idx)} wells; predicted {len(test_idx)} held-out wells.')

Trained on 6895 wells; predicted 1818 held-out wells.


In [7]:
"""7. Evaluate against the training-median baseline and build diagnostic tables."""
test_targets = data.iloc[test_idx][TARGET_COLUMNS].to_numpy()
baseline = np.tile(data.iloc[train_idx][TARGET_COLUMNS].median().to_numpy(), (len(test_idx), 1))
metrics = {
    'data_path': str(DATA_PATH), 'feature_columns': feature_columns,
    'target_columns': TARGET_COLUMNS, 'group_column': GROUP_COLUMN,
    'train_rows': int(len(train_idx)), 'test_rows': int(len(test_idx)),
    'train_groups': int(groups.iloc[train_idx].nunique()),
    'test_groups': int(groups.iloc[test_idx].nunique()),
}
for index, target in enumerate(TARGET_COLUMNS):
    metrics[f'model_mae_{target}'] = float(mean_absolute_error(test_targets[:, index], test_predictions[:, index]))
    metrics[f'model_rmse_{target}'] = float(mean_squared_error(test_targets[:, index], test_predictions[:, index]) ** 0.5)
    metrics[f'baseline_mae_{target}'] = float(mean_absolute_error(test_targets[:, index], baseline[:, index]))

predictions_df = data.iloc[test_idx][IDENTIFIER_COLUMNS + TARGET_COLUMNS].copy()
predictions_df['predicted_x_gate'] = test_predictions[:, 0]
predictions_df['predicted_y_gate'] = test_predictions[:, 1]
predictions_df['x_gate_error'] = predictions_df['predicted_x_gate'] - predictions_df['x_gate']
predictions_df['y_gate_error'] = predictions_df['predicted_y_gate'] - predictions_df['y_gate']
forest = model.named_steps['randomforestregressor']
importance_df = pd.DataFrame({'feature': feature_columns, 'importance': forest.feature_importances_}).sort_values('importance', ascending=False)
display(pd.DataFrame({
    'target': TARGET_COLUMNS,
    'model_mae': [metrics[f'model_mae_{target}'] for target in TARGET_COLUMNS],
    'model_rmse': [metrics[f'model_rmse_{target}'] for target in TARGET_COLUMNS],
    'baseline_mae': [metrics[f'baseline_mae_{target}'] for target in TARGET_COLUMNS],
}))
display(importance_df.head(15))
display(predictions_df.head())

,target,model_mae,model_rmse,baseline_mae
0,x_gate,356.795007,695.854236,803.375448
1,y_gate,335.527403,504.324905,1441.492164


,feature,importance
3,y_max_ch1,0.406627
2,y_min_ch1,0.393437
1,x_max_ch2,0.070711
7,ch1_std,0.024613
5,amplitude_range_MUT,0.014545
4,amplitude_range_WT,0.010962
0,x_min_ch2,0.009840
9,ch1_cv,0.009659
8,ch1_iqr,0.008959
13,ch1_log_std,0.008193


,dataset,parent_dataset,well,x_gate,y_gate,predicted_x_gate,predicted_y_gate,x_gate_error,y_gate_error
288,210121_TRT13_TRT008_(F1-R1)_A01_Amplitude,210121_TRT13_TRT008_(F1-R1),A01,1774.882744,2489.974395,1876.717552,2210.359851,101.834808,-279.614544
289,210121_TRT13_TRT008_(F1-R1)_A02_Amplitude,210121_TRT13_TRT008_(F1-R1),A02,1915.414721,1998.315109,2260.221093,2244.920632,344.806372,246.605523
290,210121_TRT13_TRT008_(F1-R1)_A03_Amplitude,210121_TRT13_TRT008_(F1-R1),A03,1971.694566,2012.608398,2011.474706,2331.337017,39.780140,318.728619
291,210121_TRT13_TRT008_(F1-R1)_A04_Amplitude,210121_TRT13_TRT008_(F1-R1),A04,1616.151372,2057.297660,2016.929779,2118.612121,400.778407,61.314461
292,210121_TRT13_TRT008_(F1-R1)_A05_Amplitude,210121_TRT13_TRT008_(F1-R1),A05,2152.262607,1999.763841,1969.930285,2117.538536,-182.332322,117.774695


In [8]:
"""8. Save the model and evaluation artifacts."""
joblib.dump(model, MODEL_PATH)
with METRICS_PATH.open('w', encoding='utf-8') as metrics_file:
    json.dump(metrics, metrics_file, indent=2)
predictions_df.to_csv(TEST_PREDICTIONS_PATH, index=False)
importance_df.to_csv(IMPORTANCE_PATH, index=False)
for path in [MODEL_PATH, METRICS_PATH, TEST_PREDICTIONS_PATH, IMPORTANCE_PATH]:
    print(f'Saved: {path}')
for target in TARGET_COLUMNS:
    print(f"{target} MAE: {metrics[f'model_mae_{target}']:.3f}; baseline: {metrics[f'baseline_mae_{target}']:.3f}")

Saved: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\output\biorad_gate_model.joblib
Saved: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\output\biorad_gate_model_metrics.json
Saved: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\output\biorad_gate_model_predictions.csv
Saved: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\output\biorad_gate_feature_importance.csv
x_gate MAE: 356.795; baseline: 803.375
y_gate MAE: 335.527; baseline: 1441.492


In [9]:
"""9. Load inference data and validate required features."""
inference_data = pd.read_csv(INFERENCE_DATA_PATH)
with METRICS_PATH.open('r', encoding='utf-8') as metrics_file:
    saved_metrics = json.load(metrics_file)
inference_features = saved_metrics['feature_columns']
missing_columns = [column for column in inference_features if column not in inference_data]
if missing_columns:
    raise ValueError(f'Required feature columns are missing: {missing_columns}')
print(f'Inference wells: {len(inference_data)}; required features: {len(inference_features)}')

Inference wells: 8713; required features: 28


In [10]:
"""10. Run batch prediction with the saved model and export the results."""
saved_model = joblib.load(MODEL_PATH)
batch_predictions = saved_model.predict(inference_data[inference_features])
result = inference_data.copy()
result['predicted_x_gate'] = batch_predictions[:, 0]
result['predicted_y_gate'] = batch_predictions[:, 1]
assert np.isfinite(batch_predictions).all()
assert len(result) == len(inference_data)
result.to_csv(PREDICTION_OUTPUT_PATH, index=False)
print(f'Saved predictions: {PREDICTION_OUTPUT_PATH}\nPredicted wells: {len(result)}')
preview_columns = [column for column in IDENTIFIER_COLUMNS if column in result] + ['predicted_x_gate', 'predicted_y_gate']
display(result[preview_columns].head(10))

Saved predictions: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\output\biorad_gate_predictions_all_wells.csv
Predicted wells: 8713


,dataset,parent_dataset,well,predicted_x_gate,predicted_y_gate
0,Rapeseed TRT-13_TRT001_A01_Amplitude,Rapeseed TRT-13_TRT001,A01,1749.542183,3334.058153
1,Rapeseed TRT-13_TRT001_A02_Amplitude,Rapeseed TRT-13_TRT001,A02,1651.363479,2768.152399
2,Rapeseed TRT-13_TRT001_A03_Amplitude,Rapeseed TRT-13_TRT001,A03,1725.684726,3697.016958
3,Rapeseed TRT-13_TRT001_A04_Amplitude,Rapeseed TRT-13_TRT001,A04,1725.911876,3054.919228
4,Rapeseed TRT-13_TRT001_A05_Amplitude,Rapeseed TRT-13_TRT001,A05,1690.676528,2679.514761
5,Rapeseed TRT-13_TRT001_A06_Amplitude,Rapeseed TRT-13_TRT001,A06,1708.214389,2677.270143
6,Rapeseed TRT-13_TRT001_A07_Amplitude,Rapeseed TRT-13_TRT001,A07,1584.801489,2992.996390
7,Rapeseed TRT-13_TRT001_A08_Amplitude,Rapeseed TRT-13_TRT001,A08,1629.289857,2674.460006
8,Rapeseed TRT-13_TRT001_A09_Amplitude,Rapeseed TRT-13_TRT001,A09,1597.805817,2878.991060
9,Rapeseed TRT-13_TRT001_A10_Amplitude,Rapeseed TRT-13_TRT001,A10,1683.255163,3447.863610
